# Domain 8: Tools and MCPs
**Exam**: Claude Certified Developer – Foundations (CCDV-F)
**Weighting**: 10.6% of exam
**Depth**: deep

## Domain Overview

This domain covers tool implementation practices (function calling, error handling, usage
patterns), MCP server development (authoring, deployment, resources/tools/prompts, communication
patterns), and agentic customization tradeoffs among built-in tools, custom tools, Skills, and MCPs.

**Why it matters**: developers frequently need to decide not just how to implement a tool, but which
kind of extensibility mechanism (built-in tool, custom tool, Skill, or MCP server) actually fits.

**Key concepts to know**: tool implementation patterns, MCP server authoring/communication patterns,
and the built-in/custom-tool/Skill/MCP tradeoff.

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Tool Implementation: function calling, configuration, error handling, usage patterns

Tool implementation should define a clear function schema, validate inputs before
executing side effects, return structured (not just plain-text) errors, and choose the right dispatch
pattern: agentic harness dispatch (framework handles the loop) vs. manual dispatch (you write the
loop), and client-side vs. server-side execution, plus whether an action needs human approval before
running.

In [ ]:
tool_schema = {
    "name": "process_refund",
    "description": "process_refund(customer_id: str, amount: float) -> RefundResult. Requires a verified customer_id.",
    "input_schema": {
        "type": "object",
        "properties": {"customer_id": {"type": "string"}, "amount": {"type": "number"}},
        "required": ["customer_id", "amount"],
    },
}
def execute_process_refund(customer_id: str, amount: float) -> dict:
    if amount <= 0:
        return {"isError": True, "errorCategory": "validation", "isRetryable": False, "message": "amount must be positive"}
    return {"isError": False, "result": f"refunded {amount} to {customer_id}"}

print(execute_process_refund("C1", -5))
print(execute_process_refund("C1", 50))

### MCP Server Development: authoring, resources/tools/prompts, communication patterns

An MCP server exposes tools (actions), resources (readable content/catalogs), and prompts
(reusable prompt templates) over a standard protocol, communicating via stdio (local process) or
sockets (networked), and can be consumed by any MCP-compatible client, not just one application.

In [ ]:
mcp_server_capabilities = {
    "tools": ["process_refund", "lookup_order"],
    "resources": ["order-status-catalog", "refund-policy-doc"],
    "prompts": ["refund-decision-template"],
    "transport": "stdio",  # or "sockets" for networked deployments
}
print(mcp_server_capabilities)

### Agentic Customization: built-in tools vs. custom tools vs. Skills vs. MCPs

Built-in tools (Read/Write/Bash/Grep/Glob) suit generic file/shell operations. Custom
tools suit one-off, app-specific functions. Skills suit packaged, on-demand workflows within Claude
Code. MCP servers suit capabilities that must be reused and independently maintained across
multiple Claude applications.

In [ ]:
def choose_extensibility_mechanism(reused_across_apps: bool, is_generic_file_or_shell_op: bool, is_ondemand_workflow_in_claude_code: bool) -> str:
    if reused_across_apps:
        return "MCP server"
    if is_generic_file_or_shell_op:
        return "built-in tool"
    if is_ondemand_workflow_in_claude_code:
        return "Skill"
    return "custom tool (app-specific function)"

print(choose_extensibility_mechanism(True, False, False))
print(choose_extensibility_mechanism(False, True, False))
print(choose_extensibility_mechanism(False, False, True))

## Mini Hands-On Project

**Covers**: all Domain 8 sub-objectives.

**Scenario**: Decide how to expose an internal inventory-check capability across 3 Claude
applications and implement it with proper error handling.

### Step 1: Choose the extensibility mechanism

In [ ]:
print(choose_extensibility_mechanism(reused_across_apps=True, is_generic_file_or_shell_op=False, is_ondemand_workflow_in_claude_code=False))

### Step 2: Design the MCP server capabilities

In [ ]:
print({"tools": ["check_inventory"], "resources": ["inventory-catalog"], "prompts": [], "transport": "stdio"})

### Step 3: Implement the tool with structured error handling

In [ ]:
def execute_check_inventory(sku: str) -> dict:
    if not sku.startswith("SKU-"):
        return {"isError": True, "errorCategory": "validation", "isRetryable": False, "message": "sku must match SKU-#####"}
    return {"isError": False, "stock": 12}

print(execute_check_inventory("bad"))
print(execute_check_inventory("SKU-1"))

## Lab Exercises

### Exercise 1: Tool schema and validation design

**Objective**: Design a schema and validated executor for a 'schedule_callback' tool.

**Produce**: A schema dict and an executor function with validation.

In [ ]:
# TODO: design tool_schema and execute_schedule_callback(customer_id, callback_time) with
# validation that callback_time is not in the past (simulate with a simple string comparison)
schedule_callback_schema = {}  # TODO

def execute_schedule_callback(customer_id: str, callback_time: str) -> dict:
    pass

print(execute_schedule_callback("C1", "2020-01-01"))  # should fail validation (in the past, illustratively)
print(execute_schedule_callback("C1", "2030-01-01"))

# Expected output: The past-dated callback is rejected with a structured validation error; the future one succeeds.

In [ ]:
# STRETCH GOAL: add a retryable transient-error simulation (e.g., 10% random chance of a
# "calendar service unavailable" error) and handle it with isRetryable: true.
import random
def execute_schedule_callback_v2(customer_id: str, callback_time: str) -> dict:
    pass

# Expected output: A simulated transient failure correctly marked isRetryable: true, distinct from the validation error case.

### Exercise 2: MCP server capability design

**Objective**: Design MCP server capabilities for a 'documentation-search' server used by 4 internal apps.

**Produce**: A capabilities dict with at least 2 tools, 1 resource, and a transport choice.

In [ ]:
# TODO: design mcp_server_capabilities for a documentation-search server (used by 4 apps, so
# consider whether stdio or sockets fits better)
doc_search_server = {}  # TODO
print(doc_search_server)

# Expected output: A capabilities dict with sensible tools/resources and sockets chosen given multi-app networked use.

In [ ]:
# STRETCH GOAL: sketch the .mcp.json entry that a consuming Claude Code project would use to
# connect to this server.
mcp_json_entry = {}  # TODO
print(mcp_json_entry)

# Expected output: A plausible .mcp.json snippet referencing this server's connection details.

### Exercise 3: Extensibility mechanism decision batch

**Objective**: Choose the right mechanism for 5 new capabilities.

**Produce**: 5 choices with reasoning.

In [ ]:
capabilities = [
    ("Renaming a batch of local files", False, True, False),
    ("A one-off calculation specific to this single app", False, False, False),
    ("A packaged 'generate release notes' workflow used on-demand in Claude Code", False, False, True),
    ("A CRM lookup tool shared by the sales, support, and marketing Claude apps", True, False, False),
    ("Searching file contents for a function name", False, True, False),
]
# TODO: call choose_extensibility_mechanism for each
for desc, reused, generic_file_op, ondemand_skill in capabilities:
    pass

# Expected output: Correctly maps to: built-in tool, custom tool, Skill, MCP server, built-in tool respectively.

In [ ]:
# STRETCH GOAL: for the CRM lookup case, list one risk of choosing "custom tool per app" instead
# of a shared MCP server.
risk_of_wrong_choice = "..."  # TODO

# Expected output: A risk such as duplicated logic and inconsistent behavior/maintenance across the 3 apps if built as separate custom tools.

### Exercise 4: Structured error consistency check

**Objective**: Audit 3 tool executors for inconsistent error response shapes and standardize them.

**Produce**: 3 standardized executors returning a consistent error shape.

In [ ]:
def executor_a(x):
    if x < 0:
        return "error: negative value"  # inconsistent: plain string, not structured
    return {"isError": False, "result": x}

def executor_b(x):
    if x < 0:
        raise ValueError("negative value")  # inconsistent: raises instead of returning
    return {"isError": False, "result": x}

# TODO: write standardize(executor, x) that calls the executor and normalizes any error shape
# (string, exception, or already-structured) into the consistent {"isError", "errorCategory", "isRetryable", "message"} shape
def standardize(executor, x):
    pass

print(standardize(executor_a, -1))
print(standardize(executor_b, -1))

# Expected output: Both inconsistent executors produce the same standardized structured error shape when wrapped.

In [ ]:
# STRETCH GOAL: apply standardize() to a correctly-structured executor (like execute_process_refund)
# and verify it passes through unchanged.
print(standardize(lambda x: execute_process_refund("C1", x), -5))

# Expected output: The already-correctly-structured error passes through the standardization wrapper unchanged.

## Self-Assessment: Tools and MCPs

Answer each question, then check the Answers cell below.


**Q1.** Why should a tool executor validate inputs before performing any side effect?

**Q2.** What are the three capability types an MCP server can expose?

**Q3.** When should a capability be built as an MCP server rather than an app-specific custom tool?

**Q4.** What is the difference between built-in tools and Skills in Claude Code?

**Q5.** Why is a consistent, structured error response shape important across all of an application's tools?

## Answers


**A1.** To prevent invalid or malicious input from causing an unintended side effect (e.g., a negative refund amount) before it's too late to stop.

**A2.** Tools (actions), resources (readable content/catalogs), and prompts (reusable prompt templates).

**A3.** When the capability needs to be reused and independently maintained across multiple Claude applications, not just one.

**A4.** Built-in tools are generic, pre-provided file/shell operations (Read, Write, Bash, Grep, Glob); Skills are packaged, on-demand workflows specific to a project or team.

**A5.** So that calling code (and the model) can reliably distinguish and handle different error categories (retryable vs. not, validation vs. business) the same way everywhere.

---
**Next**: [Capstone Project](./00-capstone.ipynb)
**Back**: [README](./README.md)